# QR Code Generator - Random UID / No Noise

This notebook generates clean QR codes for the English-German Vocabulary Book. Each QR code contains a random `uid` URL parameter. Keep the generated CSV file because it is the mapping table for users.

## 1. Install libraries

In [ ]:
!pip install -q "qrcode[pil]" opencv-python

## 2. Import libraries

In [ ]:
import os
import zipfile
import secrets
from pathlib import Path
from urllib.parse import urlsplit, urlunsplit, parse_qsl, urlencode

import pandas as pd
import qrcode
from PIL import Image
import matplotlib.pyplot as plt

try:
    import cv2
    CV2_AVAILABLE = True
except Exception:
    CV2_AVAILABLE = False

from google.colab import files

## 3. Settings

Change `BASE_URL` to your GitHub Pages URL. Change `USER_COUNT` to the number of QR codes you want.

In [ ]:
# Change this URL to your own GitHub Pages URL.
# Example: "https://bokuhabobu.github.io/present/"
BASE_URL = "https://bokuhabobu.github.io/present/"

# Number of random QR codes to create.
USER_COUNT = 20

# Random ID settings.
# TOKEN_BYTES=6 makes 12 hex characters, e.g. u_a3f91c7d20aa
UID_PREFIX = "u_"
TOKEN_BYTES = 6

# Output names.
OUTPUT_DIR = Path("qr_codes_random")
MAPPING_CSV = "qr_mapping_random.csv"
ZIP_FILENAME = "qr_codes_random.zip"

OUTPUT_DIR.mkdir(exist_ok=True)

## 4. Helper functions

In [ ]:
def sanitize_uid(uid):
    """Keep only safe characters for filename and URL parameter."""
    safe = "".join(
        ch for ch in str(uid).strip()
        if ch.isalnum() or ch in ["_", "-"]
    )
    return safe[:60] or "default"


def generate_random_uids(count, prefix="u_", token_bytes=6):
    """
    Generate unique random user IDs.

    Important:
    - Run this when issuing NEW QR codes.
    - For reissuing an existing user's QR code, reuse the uid from the mapping CSV.
    """
    uid_set = set()

    while len(uid_set) < count:
        uid = f"{prefix}{secrets.token_hex(token_bytes)}"
        uid = sanitize_uid(uid)
        uid_set.add(uid)

    return sorted(uid_set)


def make_user_url(base_url, uid):
    """Create a GitHub Pages URL with uid parameter safely."""
    base_url = base_url.strip()
    parts = urlsplit(base_url)

    query_pairs = parse_qsl(parts.query, keep_blank_values=True)
    query_pairs = [(key, value) for key, value in query_pairs if key != "uid"]
    query_pairs.append(("uid", uid))

    new_query = urlencode(query_pairs)
    return urlunsplit((parts.scheme, parts.netloc, parts.path, new_query, parts.fragment))


def create_clean_qr(url, save_path):
    """
    Create a readable QR code without noise.

    Settings:
    - black / white only
    - high error correction
    - large modules
    - enough quiet zone
    - RGB PNG output
    """
    qr = qrcode.QRCode(
        version=None,
        error_correction=qrcode.constants.ERROR_CORRECT_H,
        box_size=12,
        border=5,
    )

    qr.add_data(url)
    qr.make(fit=True)

    img = qr.make_image(fill_color="black", back_color="white")
    img = img.convert("RGB")
    img.save(save_path)

    return img


def read_qr_with_opencv(image_path):
    """Check whether OpenCV can decode the generated QR image."""
    if not CV2_AVAILABLE:
        return "not_checked", "OpenCV is not available"

    img = cv2.imread(str(image_path))
    detector = cv2.QRCodeDetector()
    decoded_text, points, _ = detector.detectAndDecode(img)

    if decoded_text:
        return "ok", decoded_text

    return "failed", "Could not decode"

## 5. Generate random user IDs

In [ ]:
USER_IDS = generate_random_uids(
    count=USER_COUNT,
    prefix=UID_PREFIX,
    token_bytes=TOKEN_BYTES,
)

print("Generated user IDs:")
for uid in USER_IDS:
    print(uid)

## 6. Generate QR codes

In [ ]:
records = []

for uid in USER_IDS:
    url = make_user_url(BASE_URL, uid)

    filename = f"{uid}.png"
    filepath = OUTPUT_DIR / filename

    create_clean_qr(url, filepath)
    check_status, decoded_result = read_qr_with_opencv(filepath)

    records.append({
        "uid": uid,
        "url": url,
        "qr_file": filename,
        "decode_check": check_status,
        "decoded_text": decoded_result,
        "assigned_to": "",
        "note": "",
    })

print(f"Generated {len(records)} QR codes.")

## 7. Preview QR codes

In [ ]:
preview_count = min(len(records), 6)

for i in range(preview_count):
    row = records[i]
    img = Image.open(OUTPUT_DIR / row["qr_file"])

    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.axis("off")
    plt.title(row["uid"])
    plt.show()

## 8. Save mapping CSV

Keep this CSV file. It is the user-to-QR management table.

In [ ]:
mapping_df = pd.DataFrame(records)
mapping_df.to_csv(MAPPING_CSV, index=False, encoding="utf-8-sig")

print("Generated QR mapping:")
display(mapping_df)

## 9. Zip QR images and CSV

In [ ]:
with zipfile.ZipFile(ZIP_FILENAME, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    for png_file in OUTPUT_DIR.glob("*.png"):
        zipf.write(png_file, arcname=png_file.name)

    zipf.write(MAPPING_CSV, arcname=MAPPING_CSV)

print(f"Created: {ZIP_FILENAME}")

## 10. Download ZIP

In [ ]:
files.download(ZIP_FILENAME)